# 📈 Baseline Modeling: Developer Salary Prediction

## Objective
This notebook establishes baseline predictive models for developer salary (`Salary`) using `data/cleaned_salary_data.csv`:
1. **Median Dummy Baseline**: Predict the training set median salary for all test rows.
2. **Ridge Regression Pipeline**: Fit a scikit-learn `Pipeline` with `ColumnTransformer` (OneHotEncoder for categorical features, scaling for numerical features) on log-transformed salary (`np.log`), evaluate on the original scale via `np.exp`, and compare MAE, RMSE, and $R^2$.


In [1]:
# 1. Imports and Configuration
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

print("Setup complete.")


Setup complete.


### 🔍 Setup Note
Libraries imported and numerical float formatting configured for evaluation tables.


In [2]:
# 2. Data Loading & Train-Test Split
data_path = os.path.join('..', 'data', 'cleaned_salary_data.csv')
df = pd.read_csv(data_path)

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

X = df.drop(columns=['Salary'])
y = df['Salary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


Loaded dataset: 16954 rows, 8 columns
X_train shape: (13563, 7)
X_test shape:  (3391, 7)
y_train shape: (13563,)
y_test shape:  (3391,)


### 🔍 Train/Test Split Findings
- Loaded dataset shape: 16,954 rows and 8 columns.
- With `test_size=0.2` and `random_state=42`:
  - `X_train shape`: (13563, 7)
  - `X_test shape`: (3391, 7)
  - `y_train shape`: (13563,)
  - `y_test shape`: (3391,)


In [3]:
# 3. Baseline Model: Predict Training Set Median Salary
train_median = y_train.median()
print(f"Training Set Median Salary: ${train_median:,.2f}")

y_pred_baseline = np.full_like(y_test, fill_value=train_median)

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = root_mean_squared_error(y_test, y_pred_baseline)
r2_base = r2_score(y_test, y_pred_baseline)

results = pd.DataFrame([
    {
        'Model': 'Median Baseline',
        'MAE ($)': mae_base,
        'RMSE ($)': rmse_base,
        'R-squared': r2_base
    }
])

print("\nBaseline Evaluation Results:")
display(results)


Training Set Median Salary: $83,960.00

Baseline Evaluation Results:


,Model,MAE ($),RMSE ($),R-squared
0,Median Baseline,"45,846.6697","62,496.3831",-0.0419


### 🔍 Baseline Model Findings
- **Training Set Median Salary**: $83,960.00
- **Baseline Evaluation on Test Set (3,391 rows)**:
  - `MAE ($)`: 45,846.6697
  - `RMSE ($)`: 62,496.3831
  - `R-squared`: -0.0419


In [4]:
# 4. Scikit-Learn Pipeline & Ridge Regression (Trained on np.log)
cat_cols = ['Country', 'Education', 'Role', 'RemoteWork', 'OrgSize']
num_cols = ['Experience_Years', 'Work_Experience_Years']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

# Fit ONLY on training data with log-transformed salary
y_train_log = np.log(y_train)
ridge_pipeline.fit(X_train, y_train_log)

# Predict on test set and convert predictions back with np.exp
y_pred_log = ridge_pipeline.predict(X_test)
y_pred_ridge = np.exp(y_pred_log)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

results = pd.concat([
    results,
    pd.DataFrame([{
        'Model': 'Ridge Regression (log-target)',
        'MAE ($)': mae_ridge,
        'RMSE ($)': rmse_ridge,
        'R-squared': r2_ridge
    }])
], ignore_index=True)

print("Model Comparison Results Table:")
display(results)


Model Comparison Results Table:


,Model,MAE ($),RMSE ($),R-squared
0,Median Baseline,"45,846.6697","62,496.3831",-0.0419
1,Ridge Regression (log-target),"28,516.7307","42,876.0335",0.5096


### 🔍 Ridge Regression Evaluation & Comparison Findings
- **Results Table**:
  - `Median Baseline`: MAE = $45,846.6697 | RMSE = $62,496.3831 | R-squared = -0.0419
  - `Ridge Regression (log-target)`: MAE = $28,516.7307 | RMSE = $42,876.0335 | R-squared = 0.5096

Fitting a regularized Ridge regression on log-transformed salary reduces the Mean Absolute Error from $45,846.6697 to $28,516.7307 (an improvement of $17,329.9390) and achieves an R-squared of 0.5096 on the unseen test set.
